# Grade C · Part 3 
Compare HMMs with 2, 3 and 4 hidden states. Each model observes the same four symbols. Which model assigns the highest probability to an independent sequence after training?

In [13]:
from hmm3 import forward_pass, backward_pass, reestimation
import numpy as np
from itertools import permutations

## 1. Read the two sequences
The first integer in each file gives the sequence length, the observations start at the second integer. Store the sequences separately so it is clear which one the algorithm uses for training.

In [ ]:
from pathlib import Path
from itertools import permutations
import numpy as np

def read_sequence(filename, expected_length):

    values = np.array(Path(filename).read_text().split(), dtype=int)
    length, observations = values[0], values[1:]
    
    return observations

train = read_sequence("hmm_c_N10000.in", 10000)
evaluation = read_sequence("hmm_c_N1000.in", 1000)
print("Training:", len(train), "| Evaluation:", len(evaluation))

Training: 10000 | Evaluation: 1000


## 2. Initialize the models consistently
For $K$ states, $A$ has shape $K × K$, $B$ has shape $K × 4$, and $π$ contains $K$ values. Each row of $A$ and $B$ represents a probability distribution.

The initialize(k) function draws random values from a uniform distribution between 0.1 and 1, It then divides each row of A and B by its row sum and  divides pi by the sum of its entries, so the probabilities in each row add up to 1. The fixed random seed (42) ensures that each run produces the same initial values for a given number of hidden states.

At the end, print the exact initial values so the experiment can be reproduced.

In [5]:
SEED = 42
MAX_ITERS = 100

def normalize_rows(matrix):
    return matrix / matrix.sum(axis=1, keepdims=True)

def initialize(k):
    rng = np.random.default_rng(SEED)
    A = normalize_rows(rng.uniform(0.1, 1, (k, k)))
    B = normalize_rows(rng.uniform(0.1, 1, (k, 4)))
    pi = rng.uniform(0.1, 1, k) 
    return A, B, pi / pi.sum()

#Model with 2 hidden states
A2, B2, pi2 = initialize(2)
print("K = 2")
print("A0 =", A2.tolist())
print("B0 =", B2.tolist())
print("pi0 =", pi2.tolist())

#Model with 3 hidden states
A3, B3, pi3 = initialize(3)
print("\nK = 3")
print("A0 =", A3.tolist())
print("B0 =", B3.tolist())
print("pi0 =", pi3.tolist())

#Model with 4 hidden states
A4, B4, pi4 = initialize(4)
print("\nK = 4")
print("A0 =", A4.tolist())
print("B0 =", B4.tolist())
print("pi0 =", pi4.tolist())

#Store the initial models for training
starts = {
    2: (A2, B2, pi2),
    3: (A3, B3, pi3),
    4: (A4, B4, pi4),
}

K = 2
A0 = [[0.6167471662775277, 0.38325283372247243], [0.5453354412834304, 0.45466455871656963]]
B0 = [[0.06705599731155627, 0.35497366248357465, 0.28491444898365015, 0.29305589122121894], [0.10309158338678553, 0.24197170799659815, 0.20767406842932515, 0.4472626401872912]]
pi0 = [0.4470359700908265, 0.5529640299091735]

K = 3
A0 = [[0.3680471425443695, 0.22870816119828832, 0.4032446962573422], [0.38489822967886794, 0.09773308984257131, 0.5173686804785608], [0.4342470766231675, 0.44665570491061357, 0.11909721846621882]]
B0 = [[0.1979710403469074, 0.16991015900340856, 0.3659314178475254, 0.26618738280215865], [0.37468156247583134, 0.22248257619649134, 0.13575023741781236, 0.2670856239098648], [0.06417870502625027, 0.3444110583209961, 0.27251366448363995, 0.31889657216911366]]
pi0 = [0.1824826634231775, 0.42395982013386235, 0.39355751644296]

K = 4
A0 = [[0.2754434200459781, 0.17116328543506892, 0.3017849764697845, 0.2516083180491684], [0.06705599731155627, 0.35497366248357465, 0.2849144

## 3. Train and evaluate each model
Start with the log-likelihood of the initial model. Accept an update if it increases the training log-likelihood. Stop at the first update that gives no increase or after 100 updates. Apply the same rule to all three models. On the independent sequence of 1,000 observations, run only the forward algorithm with the learned parameters fixed. Divide each log-likelihood by the length of its own sequence.

In [10]:
def train_model(initial_model, observations, max_iters=100):

    A, B, pi = initial_model

    c = [0.0] * len(observations)
    alpha = forward_pass(A, B, c, pi, observations)
    log_prob = -np.log(c).sum()

    for i in range(max_iters):
        beta = backward_pass(A, B, c, observations)
        new_A, new_B = reestimation(A, B, alpha, beta, observations)

        #Estimate pi from the state probabilities at the first time step (cause reestimation gives just new A,B).
        gamma0 = np.array(alpha[0]) * np.array(beta[0])
        new_pi = gamma0 / gamma0.sum()

        #Check the likelihood of the updated model before accepting it.
        new_c = [0.0] * len(observations)
        new_alpha = forward_pass(new_A, new_B, new_c, new_pi, observations)
        if 0.0 in new_c:
            break
        
        #Accept update or stops (convergence crit)
        new_log_prob = -np.log(new_c).sum()
        if new_log_prob <= log_prob:
            break

        A, B, pi = new_A, new_B, new_pi
        alpha, c, log_prob = new_alpha, new_c, new_log_prob

    return np.asarray(A), np.asarray(B), np.asarray(pi)

Apply the same rule to all three models.

In [11]:
models = {}
models[2] = train_model(starts[2], train)
models[3] = train_model(starts[3], train)
models[4] = train_model(starts[4], train)

We compare the HMMs with 2, 3, and 4 hidden states. For each model, we use the forward algorithm to compute the log-likelihood of the training sequence (10,000 observations) and the independent evaluation sequence (1,000 observations), keeping the learned parameters fixed.

We divide then each log-likelihood by the number of observations because the sequences have different lengths (train,eval). A higher evaluation log-likelihood per observation indicates that the model predicts the independent sequence better.

In [ ]:
def score_per_observation(model, observations):
    A, B, pi = model
    c = [0.0] * len(observations)
    #Use forward pass with learned parameters
    forward_pass(A, B, c, pi, observations)

    if 0.0 in c:
        return -np.inf

    log_likelihood = -np.log(c).sum()
    return log_likelihood / len(observations)


def print_scores(k):
    train_score = score_per_observation(models[k], train)
    eval_score = score_per_observation(models[k], evaluation)
    print(f"{k:>6}  {train_score:>14.6f}  {eval_score:>14.6f}")


print("States   Train logL/T     Eval logL/T") #print as suggested in the assignment
print_scores(2)
print_scores(3)
print_scores(4)

States   Train logL/T     Eval logL/T
     2       -1.346288       -1.352421
     3       -1.381312       -1.388747
     4       -1.341270       -1.347738


 ## 4. Interpret the result

The four-state model achieved the highest log-likelihood per observation on both the training and evaluation sequences. On the independent evaluation sequence, it scored −1.347738, slightly better than the two-state model (−1.352421). The three-state model had the lowest score (−1.388747), even though the data were generated by a three-state HMM.

Training likelihood alone is insufficient for model selection because a more flexible model can fit patterns specific to the training data (overfit). Here, the four-state model also performed best on unseen data. However, its advantage over the two-state model was small, and only one initialization was tested for each model size.

## 5. Compare the learned three-state HMM with the generating model

We compare the learned transition matrix A, emission matrix B, and initial state distribution π with the generating parameters. Hidden-state labels are arbitrary, so learned state 0 may correspond to a different state in the generating model. We test all six possible state orderings and choose the one with the smallest total squared difference across A, B, and π.

In [14]:
# 1.Parameters of the model that generated the data
A_true = np.array([
    [0.70, 0.05, 0.25],
    [0.10, 0.80, 0.10],
    [0.20, 0.30, 0.50]
])

B_true = np.array([
    [0.70, 0.20, 0.10, 0.00],
    [0.10, 0.40, 0.30, 0.20],
    [0.00, 0.10, 0.20, 0.70]
])

pi_true = np.array([1.0, 0.0, 0.0])

#2.Get the parameters of the trained three-state model
A_learned, B_learned, pi_learned = models[3]

#3.Try all six ways to match the learned states to the true states
best_error = float("inf")

for order in permutations([0, 1, 2]):
    order = list(order)

    #In A, both rows and columns represent hidden states.
    A_test = A_learned[order, :][:, order]

    #In B, only the rows represent hidden states.
    B_test = B_learned[order, :]
    pi_test = pi_learned[order]

    error_A = np.sum((A_test - A_true) ** 2)
    error_B = np.sum((B_test - B_true) ** 2)
    error_pi = np.sum((pi_test - pi_true) ** 2)
    total_error = error_A + error_B + error_pi

    if total_error < best_error:
        best_error = total_error
        best_order = order
        A_best = A_test
        B_best = B_test
        pi_best = pi_test

#4. Compare the parameters after matching the states
print("Learned states matching true states 0, 1, 2:", best_order)

print("\nLearned A:")
print(A_best.round(3))
print("True A:")
print(A_true)

print("\nLearned B:")
print(B_best.round(3))
print("True B:")
print(B_true)

print("\nLearned pi:", pi_best.round(3))
print("True pi:   ", pi_true)

Learned states matching true states 0, 1, 2: [0, 1, 2]

Learned A:
[[0.414 0.228 0.359]
 [0.385 0.142 0.473]
 [0.413 0.449 0.138]]
True A:
[[0.7  0.05 0.25]
 [0.1  0.8  0.1 ]
 [0.2  0.3  0.5 ]]

Learned B:
[[0.265 0.198 0.295 0.242]
 [0.264 0.331 0.133 0.272]
 [0.263 0.309 0.164 0.264]]
True B:
[[0.7 0.2 0.1 0. ]
 [0.1 0.4 0.3 0.2]
 [0.  0.1 0.2 0.7]]

Learned pi: [1. 0. 0.]
True pi:    [1. 0. 0.]


## 6. Interpret the results

Even after matching the hidden-state labels, the three-state model did not recover the generating parameters well. For example, the true probability of staying in state 1 was 0.80, while the learned value was only 0.142. The learned emission probabilities also showed little difference between states for symbol 0: all three values were close to 0.26, compared with 0.70, 0.10, and 0.00 in the generating model, which means that observing that symbol doesn't help to distinguish the state. The learned initial distribution matched the true distribution to the precision shown. 

This makes sense since the three-state model had the lowest evaluation log-likelihood of the three models tested. However, the three-state model was trained with one initialization (seed 42) and a limit of 100 updates. From this single run, we cannot tell whether a different starting point or more iterations would improve the parameter estimates. We could test each factor separately by changing the seed while keeping the same limit, or by increasing the limit while keeping the same seed. But still remember that an higher training likelihood does not guarantee that the learned parameters are closer to the true values.